In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub gdown')
    print("Setup complete!")


In [2]:
input_file = 'datasets/finetuning/train.csv'
benchmark_dir = 'datasets/preprocessed'
output_dir = 'models/finetuned/OpenLID_v2'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import re
import json
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
import fasttext
from huggingface_hub import hf_hub_download

MODEL_NAME = "OpenLID-v2"
MODEL_ID = "openlid_v2"


In [4]:
print("Downloading OpenLID-v2 base model from Hugging Face...")
base_model_path = hf_hub_download(repo_id="laurievb/OpenLID-v2", filename="model.bin")
base_model_dir = os.path.dirname(base_model_path)

print(f"Loading base OpenLID-v2 model from {base_model_path}...")
base_model = fasttext.load_model(base_model_path)

# Extract pretrained word vectors for transfer learning / fine-tuning
vec_file_path = os.path.join(base_model_dir, "openlid_v2.vec")
if not os.path.exists(vec_file_path):
    print("Extracting pre-trained word vectors to .vec format...")
    words = base_model.get_words()
    dim = base_model.get_dimension()
    with open(vec_file_path, "w", encoding="utf-8") as f:
        f.write(f"{len(words)} {dim}\n")
        for word in words:
            v_str = " ".join(map(str, base_model.get_word_vector(word)))
            f.write(f"{word} {v_str}\n")
    print(f"Extracted {len(words)} vectors into {vec_file_path}")


Loading base OpenLID-v2 model from C:\Users\USER\.cache\huggingface\hub\models--laurievb--OpenLID-v2\snapshots\10b538ebcc452255917c8bdfc38faa49af134cd2\model.bin...
Extracting pre-trained word vectors to .vec format...
Extracted 185286 vectors into C:\Users\USER\.cache\huggingface\hub\models--laurievb--OpenLID-v2\snapshots\10b538ebcc452255917c8bdfc38faa49af134cd2\openlid_v2.vec


In [5]:
print(f"Loading finetuning dataset from {input_file}...")
df = pd.read_csv(input_file)
print(f"Loaded {len(df)} rows.")

def clean_for_openlid(text):
    text = str(text).strip().replace("\n", " ").lower()
    text = re.sub(r"[^\w\s]|\d", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_for_openlid)
df["ft_line"] = "__label__" + df["label"].astype(str) + " " + df["clean_text"]

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

os.makedirs(output_dir, exist_ok=True)
train_ft_file = os.path.join(output_dir, "train_formatted.txt")
val_ft_file = os.path.join(output_dir, "val_formatted.txt")

with open(train_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(train_df["ft_line"].tolist()) + "\n")

with open(val_ft_file, "w", encoding="utf-8") as f:
    f.write("\n".join(val_df["ft_line"].tolist()) + "\n")

print(f"Saved {len(train_df)} training samples to {train_ft_file}")
print(f"Saved {len(val_df)} validation samples to {val_ft_file}")


Loading finetuning dataset from datasets/finetuning/train.csv...
Loaded 60285 rows.
Saved 54256 training samples to models/finetuned/OpenLID_v2\train_formatted.txt
Saved 6029 validation samples to models/finetuned/OpenLID_v2\val_formatted.txt


In [6]:
print(f"Starting fine-tuning for {MODEL_NAME}...")
finetuned_model = fasttext.train_supervised(
    input=train_ft_file,
    pretrainedVectors=vec_file_path,
    dim=base_model.get_dimension(),
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    loss="softmax"
)

save_model_path = os.path.join(output_dir, f"{MODEL_ID}_finetuned.bin")
finetuned_model.save_model(save_model_path)
print(f"Fine-tuned model successfully saved to {save_model_path}")


Starting fine-tuning for OpenLID-v2...
Fine-tuned model successfully saved to models/finetuned/OpenLID_v2\openlid_v2_finetuned.bin


In [7]:
val_texts = val_df["clean_text"].tolist()
val_true = val_df["label"].astype(str).tolist()

print(f"Evaluating fine-tuned {MODEL_NAME} on {len(val_texts)} validation samples...")
preds, _ = finetuned_model.predict(val_texts, k=1)
val_pred = [p[0].replace("__label__", "") for p in preds]

acc = accuracy_score(val_true, val_pred)
macro_f1 = f1_score(val_true, val_pred, average="macro")

print("\n" + "=" * 48)
print(f"VALIDATION FINE-TUNING RESULTS ({MODEL_NAME})")
print("=" * 48)
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Macro F1:  {macro_f1 * 100:.2f}%")
print("=" * 48)
print("\nPer-language breakdown:\n")
print(classification_report(val_true, val_pred, digits=4, zero_division=0))


Evaluating fine-tuned OpenLID-v2 on 6029 validation samples...

VALIDATION FINE-TUNING RESULTS (OpenLID-v2)
Accuracy:  98.09%
Macro F1:  97.90%

Per-language breakdown:

              precision    recall  f1-score   support

        pali     0.9833    0.9770    0.9801      2349
    sanskrit     0.9742    0.9694    0.9718      1012
     sinhala     0.9814    0.9888    0.9851      2668

    accuracy                         0.9809      6029
   macro avg     0.9796    0.9784    0.9790      6029
weighted avg     0.9809    0.9809    0.9809      6029



In [8]:
print(f"Evaluating fine-tuned {MODEL_NAME} on Sinhala script target languages across benchmark datasets in {benchmark_dir}...")

TARGET_LANGUAGES = ["sinhala", "pali", "sanskrit"]

def map_benchmark_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl in ["sin", "sin_Sinh", "sinhala", "si"]:
        return "sinhala"
    if lbl in ["pli", "pli_Sinh", "pali", "pi"]:
        return "pali"
    if lbl in ["san_Sinh", "sanskrit"] or (lbl == "san" and src in ["DCS", "SansinNT", "SiDiaC-v2"]):
        return "sanskrit"
    return None

def load_benchmark_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_benchmark_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_bench = load_benchmark_dataset(file_path)
        if df_bench.empty:
            print(f"No matching target languages found in {dataset_name}.")
            continue
        
        texts = df_bench["text"].apply(clean_for_openlid).tolist()
        print(f"\nEvaluating {len(texts)} target language samples from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_bench[["text", "label", "source"]].copy()
        results["true_label"] = df_bench["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_b = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_b = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=TARGET_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_b * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_b * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (F1 scores & metrics for target languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=TARGET_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved benchmark predictions to {out_csv}")


Evaluating fine-tuned OpenLID-v2 on Sinhala script target languages across benchmark datasets in datasets/preprocessed...

Evaluating 7047 target language samples from commonlid...
BENCHMARK RESULTS (OpenLID-v2 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  97.59%
Macro F1:  97.53%

Per-language breakdown (F1 scores & metrics for target languages):

              precision    recall  f1-score   support

     sinhala     0.9593    0.9989    0.9787      2693
        pali     0.9908    0.9594    0.9748      3027
    sanskrit     0.9779    0.9668    0.9723      1327

    accuracy                         0.9759      7047
   macro avg     0.9760    0.9750    0.9753      7047
weighted avg     0.9763    0.9759    0.9758      7047

Saved benchmark predictions to datasets\benchmark_results\openlid_v2_finetuned_commonlid.csv

Evaluating 7047 target language samples from flores_plus...
BENCHMARK RESULTS (OpenLID-v2 Finetuned on train.csv - Evaluated on flores_plus)
Accuracy:  97.59%
M

In [9]:
print(f"Evaluating fine-tuned {MODEL_NAME} across ALL benchmark languages in {benchmark_dir}...")

ALL_BENCHMARK_LANGUAGES = [
    "sinhala", "pali", "sanskrit", "sanskrit_deva", "english", "tamil",
    "hindi", "bengali", "arabic", "french", "german"
]

LABEL_MAPPING_ALL = {
    "sin": "sinhala", "sin_Sinh": "sinhala", "sinhala": "sinhala", "si": "sinhala",
    "pli": "pali", "pli_Sinh": "pali", "pli_Latn": "pali", "pali": "pali", "pi": "pali",
    "san_Sinh": "sanskrit",
    "san_Deva": "sanskrit_deva", "sa": "sanskrit_deva",
    "eng": "english", "eng_Latn": "english", "english": "english", "en": "english",
    "tam": "tamil", "tam_Taml": "tamil", "tamil": "tamil", "ta": "tamil",
    "hin": "hindi", "hin_Deva": "hindi", "hindi": "hindi", "hi": "hindi",
    "ben": "bengali", "ben_Beng": "bengali", "bengali": "bengali", "bn": "bengali",
    "arb": "arabic", "arb_Arab": "arabic", "arabic": "arabic", "ar": "arabic",
    "fra": "french", "fra_Latn": "french", "french": "french", "fr": "french",
    "deu": "german", "deu_Latn": "german", "german": "german", "de": "german"
}

def map_all_label(row):
    lbl = row.get("label")
    src = row.get("source")
    if lbl == "san":
        if src in ["DCS", "SansinNT", "SiDiaC-v2"]:
            return "sanskrit"
        else:
            return "sanskrit_deva"
    return LABEL_MAPPING_ALL.get(lbl)

def load_all_languages_dataset(file_path):
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            mapped_label = map_all_label(row)
            if mapped_label:
                row["target_label"] = mapped_label
                records.append(row)
    return pd.DataFrame(records)

benchmark_files = sorted(glob.glob(os.path.join(benchmark_dir, "*.jsonl")))
if not benchmark_files:
    print(f"No benchmark datasets found in {benchmark_dir}.")
else:
    results_dir = os.path.join("datasets", "benchmark_results")
    os.makedirs(results_dir, exist_ok=True)
    
    for file_path in benchmark_files:
        dataset_name = os.path.splitext(os.path.basename(file_path))[0]
        df_all = load_all_languages_dataset(file_path)
        if df_all.empty:
            print(f"No matching languages found in {dataset_name}.")
            continue
        
        texts = df_all["text"].apply(clean_for_openlid).tolist()
        print(f"\nEvaluating {len(texts)} samples across ALL benchmark languages from {dataset_name}...")
        preds, _ = finetuned_model.predict(texts, k=1)
        
        results = df_all[["text", "label", "source"]].copy()
        results["true_label"] = df_all["target_label"]
        results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]
        
        acc_all = accuracy_score(results["true_label"], results["predicted_label"])
        macro_f1_all = f1_score(
            results["true_label"], results["predicted_label"],
            average="macro", labels=ALL_BENCHMARK_LANGUAGES, zero_division=0
        )
        
        print("=" * 65)
        print(f"ALL LANGUAGES BENCHMARK RESULTS ({MODEL_NAME} Finetuned on train.csv - Evaluated on {dataset_name})")
        print("=" * 65)
        print(f"Accuracy:  {acc_all * 100:.2f}%")
        print(f"Macro F1:  {macro_f1_all * 100:.2f}%")
        print("=" * 65)
        print("\nPer-language breakdown (All Benchmark Languages):\n")
        print(classification_report(
            results["true_label"], results["predicted_label"],
            labels=ALL_BENCHMARK_LANGUAGES, digits=4, zero_division=0
        ))
        
        out_csv = os.path.join(results_dir, f"{MODEL_ID}_finetuned_all_langs_{dataset_name}.csv")
        results.to_csv(out_csv, index=False)
        print(f"Saved all-languages benchmark predictions to {out_csv}")


Evaluating fine-tuned OpenLID-v2 across ALL benchmark languages in datasets/preprocessed...

Evaluating 77974 samples across ALL benchmark languages from commonlid...
ALL LANGUAGES BENCHMARK RESULTS (OpenLID-v2 Finetuned on train.csv - Evaluated on commonlid)
Accuracy:  8.82%
Macro F1:  5.21%

Per-language breakdown (All Benchmark Languages):

               precision    recall  f1-score   support

      sinhala     0.0802    0.9989    0.1485      2693
         pali     0.0795    0.9594    0.1468      3027
     sanskrit     0.1622    0.9668    0.2779      1327
sanskrit_deva     0.0000    0.0000    0.0000       895
      english     0.0000    0.0000    0.0000     27461
        tamil     0.0000    0.0000    0.0000        81
        hindi     0.0000    0.0000    0.0000      3666
      bengali     0.0000    0.0000    0.0000      1886
       arabic     0.0000    0.0000    0.0000     26152
       french     0.0000    0.0000    0.0000      3233
       german     0.0000    0.0000    0.0000    

In [10]:
print("\n" + "=" * 50)
print(f"FINE-TUNING & BENCHMARKING COMPLETE FOR {MODEL_NAME}")
print(f"Fine-tuned model saved to: {save_model_path}")
print("=" * 50)



FINE-TUNING & BENCHMARKING COMPLETE FOR OpenLID-v2
Fine-tuned model saved to: models/finetuned/OpenLID_v2\openlid_v2_finetuned.bin
